# Assignment 6: Customer Segmentation with K-Means
Unsupervised learning on mall customer data (annual income, spending score, age). Run all cells top to bottom.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, adjusted_rand_score)
sns.set_style("whitegrid")
SEED = 21
FEATURES = ["Annual Income", "Spending Score", "Age"]

## 1. Generate (or load) the customer data, explore it and check for missing values
The data is simulated so that it behaves like a real mall dataset: annual income in dollars, a spending score from 1 to 100, and age in years. If you have a real file (for example Kaggle *Mall_Customers.csv*), upload it and set `USE_CSV = True`.

In [ ]:
USE_CSV = False
if USE_CSV:
    df = pd.read_csv("Mall_Customers.csv")
    df = df.rename(columns={"Annual Income (k$)": "Annual Income", "Spending Score (1-100)": "Spending Score"})
    df["Annual Income"] = df["Annual Income"] * 1000          # convert k$ to $
    df = df[["CustomerID"] + FEATURES] if "CustomerID" in df else df[FEATURES]
else:
    rng = np.random.default_rng(SEED)
    # (customers, income mean, income sd, spend mean, spend sd, age mean, age sd)
    groups = [(40, 26000, 6500, 20, 8, 45, 9),    # low income, low spending
              (40, 27000, 6500, 80, 9, 25, 5),    # low income, high spending
              (50, 58000, 9000, 50, 8, 42, 11),   # mid income, average spending
              (35, 90000, 10000, 17, 8, 43, 8),   # high income, low spending
              (35, 88000, 10000, 82, 8, 32, 6)]   # high income, high spending
    parts = [np.column_stack([rng.normal(im, isd, n), rng.normal(sm, ssd, n), rng.normal(am, asd, n)])
             for n, im, isd, sm, ssd, am, asd in groups]
    data = np.vstack(parts)
    df = pd.DataFrame(data, columns=FEATURES)
    df["Annual Income"] = df["Annual Income"].clip(15000, 140000).round(-2)
    df["Spending Score"] = df["Spending Score"].clip(1, 99).round()
    df["Age"] = df["Age"].clip(18, 70).round()
    df.insert(0, "CustomerID", range(1, len(df) + 1))
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle rows

print("Shape:", df.shape)
display(df.head())
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", int(df.duplicated().sum()))
display(df[FEATURES].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, col, colr in zip(axes, FEATURES, ["#4C72B0", "#DD8452", "#55A868"]):
    sns.histplot(df[col], bins=20, kde=True, ax=ax, color=colr); ax.set_title(col)
plt.tight_layout(); plt.savefig("fig_hist.png", dpi=200); plt.show()

plt.figure(figsize=(4.6, 3.6))
sns.heatmap(df[FEATURES].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation of features"); plt.tight_layout(); plt.savefig("fig_corr.png", dpi=200); plt.show()

plt.figure(figsize=(5.2, 4))
plt.scatter(df["Annual Income"], df["Spending Score"], c=df["Age"], cmap="viridis", s=25)
plt.colorbar(label="Age"); plt.xlabel("Annual Income ($)"); plt.ylabel("Spending Score")
plt.title("Raw data (colour = age)"); plt.tight_layout(); plt.savefig("fig_raw.png", dpi=200); plt.show()

## 2. Scale the features with StandardScaler
K-Means uses Euclidean distance. Income is measured in tens of thousands of dollars while spending score and age are in tens, so without scaling income would dominate every distance.

In [ ]:
X = df[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Raw feature std      :", X.std(axis=0).round(1))
print("Scaled feature mean  :", X_scaled.mean(axis=0).round(3))
print("Scaled feature std   :", X_scaled.std(axis=0).round(3))

## 3. Choose K: elbow (inertia), silhouette score and Davies-Bouldin index

In [ ]:
K_RANGE = range(2, 11)
inertia, sil, dbi = [], [], []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(X_scaled)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, km.labels_))
    dbi.append(davies_bouldin_score(X_scaled, km.labels_))

ks = list(K_RANGE)
# Elbow point = k whose (k, inertia) point is farthest from the straight line joining the first and last points
p = np.column_stack([ks, inertia]).astype(float)
p = (p - p.min(0)) / (p.max(0) - p.min(0))
d = np.abs((p[-1, 0]-p[0, 0])*(p[0, 1]-p[:, 1]) - (p[0, 0]-p[:, 0])*(p[-1, 1]-p[0, 1]))
k_elbow = ks[int(np.argmax(d))]
k_sil, k_dbi = ks[int(np.argmax(sil))], ks[int(np.argmin(dbi))]

k_table = pd.DataFrame({"K": ks, "Inertia": np.round(inertia, 1),
                        "Silhouette": np.round(sil, 4), "Davies-Bouldin": np.round(dbi, 4)})
display(k_table)
k_table.to_csv("k_table.csv", index=False)
print(f"Elbow suggests K={k_elbow} | best silhouette K={k_sil} | lowest Davies-Bouldin K={k_dbi}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(ks, inertia, "o-"); axes[0].axvline(k_elbow, ls="--", c="r")
axes[0].set(title="Elbow method", xlabel="K", ylabel="Inertia (WCSS)")
axes[1].plot(ks, sil, "o-", c="g"); axes[1].axvline(k_sil, ls="--", c="r")
axes[1].set(title="Silhouette score (higher is better)", xlabel="K", ylabel="Silhouette")
axes[2].plot(ks, dbi, "o-", c="purple"); axes[2].axvline(k_dbi, ls="--", c="r")
axes[2].set(title="Davies-Bouldin (lower is better)", xlabel="K", ylabel="DBI")
for a in axes: a.set_xticks(ks)
plt.tight_layout(); plt.savefig("fig_choose_k.png", dpi=200); plt.show()

## 4. Fit the final K-Means model

In [ ]:
BEST_K = k_sil          # the three criteria are compared in the report; K is set from them
final = KMeans(n_clusters=BEST_K, n_init=50, random_state=SEED).fit(X_scaled)
df["Cluster"] = final.labels_
print("Chosen K            :", BEST_K)
print("Inertia             :", round(final.inertia_, 2))
print("Silhouette score    :", round(silhouette_score(X_scaled, final.labels_), 4))
print("Davies-Bouldin index:", round(davies_bouldin_score(X_scaled, final.labels_), 4))

# Stability: do different random starts give the same clusters?
aris = [adjusted_rand_score(final.labels_, KMeans(BEST_K, n_init=1, random_state=s).fit_predict(X_scaled))
        for s in range(10)]
print("Stability (ARI vs 10 random single-start fits): min = %.3f, mean = %.3f" % (min(aris), np.mean(aris)))

## 5. Visualise the clusters in 2D with PCA, and draw the silhouette plot

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
Z = pca.fit_transform(X_scaled)
centers_2d = pca.transform(final.cluster_centers_)
print("Variance explained by PC1, PC2:", pca.explained_variance_ratio_.round(3),
      "| total =", pca.explained_variance_ratio_.sum().round(3))
print("PCA loadings:"); display(pd.DataFrame(pca.components_.T, index=FEATURES, columns=["PC1", "PC2"]).round(3))

plt.figure(figsize=(6, 4.6))
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=final.labels_, palette="tab10", s=40, legend="full")
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="X", s=220, label="Centroids")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})"); plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
plt.title(f"K-Means clusters (K={BEST_K}) in PCA space"); plt.legend(title="Cluster", fontsize=8)
plt.tight_layout(); plt.savefig("fig_pca.png", dpi=200); plt.show()

def silhouette_plot(Xs, labels, title, fname):
    k = len(np.unique(labels)); s = silhouette_samples(Xs, labels); avg = s.mean()
    plt.figure(figsize=(6, 4.6)); y_low = 10
    for c in range(k):
        sc = np.sort(s[labels == c]); y_up = y_low + len(sc)
        plt.fill_betweenx(np.arange(y_low, y_up), 0, sc, alpha=0.8, color=plt.cm.tab10(c))
        plt.text(-0.07, y_low + len(sc) / 2, f"C{c}"); y_low = y_up + 10
    plt.axvline(avg, c="red", ls="--", label=f"Average = {avg:.3f}")
    plt.xlabel("Silhouette coefficient"); plt.yticks([]); plt.title(title); plt.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(fname, dpi=200); plt.show()
    return s
s_final = silhouette_plot(X_scaled, final.labels_, f"Silhouette plot (K={BEST_K})", "fig_silhouette.png")
print("Samples with negative silhouette:", int((s_final < 0).sum()))

## 6. Profile each segment: size, mean income, spending and age

In [ ]:
profile = (df.groupby("Cluster")
             .agg(Customers=("CustomerID", "count"),
                  Avg_Income=("Annual Income", "mean"),
                  Avg_Spending=("Spending Score", "mean"),
                  Avg_Age=("Age", "mean"))
             .round(1))
profile["Share_%"] = (profile["Customers"] / len(df) * 100).round(1)

# Automatic label from income and spending relative to the overall averages
inc_m, spd_m = df["Annual Income"].mean(), df["Spending Score"].mean()
def label(r):
    inc = "High income" if r.Avg_Income > inc_m * 1.2 else ("Low income" if r.Avg_Income < inc_m * 0.8 else "Mid income")
    spd = "high spending" if r.Avg_Spending > spd_m + 15 else ("low spending" if r.Avg_Spending < spd_m - 15 else "average spending")
    return f"{inc}, {spd}"
profile["Segment"] = profile.apply(label, axis=1)
display(profile[["Segment", "Customers", "Share_%", "Avg_Income", "Avg_Spending", "Avg_Age"]])
profile.to_csv("profile.csv")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, col, t in zip(axes, FEATURES, ["Annual income ($)", "Spending score", "Age"]):
    sns.boxplot(data=df, x="Cluster", y=col, palette="tab10", ax=ax); ax.set_title(t)
plt.tight_layout(); plt.savefig("fig_profile.png", dpi=200); plt.show()

plt.figure(figsize=(6, 4.6))
sns.scatterplot(data=df, x="Annual Income", y="Spending Score", hue="Cluster", palette="tab10", s=40)
plt.title("Segments on the original income-spending axes"); plt.tight_layout()
plt.savefig("fig_segments.png", dpi=200); plt.show()

## 7. How does scaling change the clusters?
We cluster the raw (unscaled) data with the same K, then compare it with the scaled result.

In [ ]:
raw_km = KMeans(n_clusters=BEST_K, n_init=50, random_state=SEED).fit(X)
df["Cluster_unscaled"] = raw_km.labels_

# Compare both labelings in the same (scaled) space so the scores are fair
cmp = pd.DataFrame({
    "Scaled": [silhouette_score(X_scaled, final.labels_), davies_bouldin_score(X_scaled, final.labels_)],
    "Unscaled": [silhouette_score(X_scaled, raw_km.labels_), davies_bouldin_score(X_scaled, raw_km.labels_)]},
    index=["Silhouette (scaled space)", "Davies-Bouldin (scaled space)"]).round(4)
display(cmp); cmp.to_csv("scaling_cmp.csv")

ari = adjusted_rand_score(final.labels_, raw_km.labels_)
print("Adjusted Rand Index between scaled and unscaled clusterings:", round(ari, 4))

# Which feature drives the unscaled clusters? (share of between-cluster variance per feature)
def eta2(col):
    tot = ((df[col] - df[col].mean()) ** 2).sum()
    btw = sum(len(g) * (g[col].mean() - df[col].mean()) ** 2 for _, g in df.groupby("Cluster_unscaled"))
    return btw / tot
eta = pd.Series({c: eta2(c) for c in FEATURES}).round(3)
print("\nVariance explained by the UNSCALED clusters, per feature (1 = fully separated):"); print(eta)
eta.to_csv("eta_unscaled.csv")

print("\nUnscaled cluster profile:")
unprof = df.groupby("Cluster_unscaled").agg(Customers=("CustomerID", "count"), Avg_Income=("Annual Income", "mean"),
         Avg_Spending=("Spending Score", "mean"), Avg_Age=("Age", "mean")).round(1)
display(unprof); unprof.to_csv("unscaled_profile.csv")
print("Cross-tab (rows = scaled cluster, columns = unscaled cluster):")
display(pd.crosstab(df["Cluster"], df["Cluster_unscaled"]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
for ax, lab, t in zip(axes, [final.labels_, raw_km.labels_], ["With StandardScaler", "Without scaling"]):
    ax.scatter(Z[:, 0], Z[:, 1], c=lab, cmap="tab10", s=30); ax.set_title(t); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.savefig("fig_scaling.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, lab, t in zip(axes, ["Cluster", "Cluster_unscaled"], ["With StandardScaler", "Without scaling"]):
    sns.scatterplot(data=df, x="Annual Income", y="Spending Score", hue=lab, palette="tab10", s=35, ax=ax, legend=False)
    ax.set_title(t)
plt.tight_layout(); plt.savefig("fig_scaling_raw.png", dpi=200); plt.show()